# 01 — Data Preprocessing

**Project:** Predictive Analytics for Enterprise Streaming Acquisitions  
**Course:** CMPS 451 — Data Mining, Big Data & Analytics (Spring 2026)  
**Team:** 11

---

## Objectives
1. Load all 7 IMDb TSV datasets using PySpark
2. Clean and filter data (remove adult content, irrelevant title types, missing values)
3. Filter unreliable ratings (numVotes < 100) per instructor feedback
4. Join all tables into a unified dataset
5. Integrate the IEEE DataPort User Ratings dataset (Dataset.npy)
6. Save the preprocessed data for downstream analysis

In [ ]:

# ── Imports & Environment Setup ──
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import findspark

os.environ['HADOOP_HOME'] = r'A:\hadoop'
os.environ['SPARK_HOME'] = r'A:\spark'
os.environ['PATH'] = r'A:\hadoop\bin;' + os.environ.get('PATH', '')
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType
)

# Paths
DATA_DIR = os.path.join("Dataset")
OUTPUT_DIR = os.path.join("outputs")
os.makedirs(os.path.join(OUTPUT_DIR, "parquet"), exist_ok=True)

print("Setup complete.")

Setup complete.


In [11]:
# ── Initialize Spark (pseudo-distributed mode) ──
spark = (
    SparkSession.builder
    .appName("IMDb_Preprocessing")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Running in pseudo-distributed mode with {spark.sparkContext.defaultParallelism} cores")

Spark version: 3.5.1
Running in pseudo-distributed mode with 12 cores


## 1. Load IMDb Datasets

In [12]:
def load_tsv(filename):
    """Load a gzipped TSV file into a Spark DataFrame."""
    path = os.path.join(DATA_DIR, filename)
    df = (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .option("nullValue", "\\N")
        .option("quote", "")
        .csv(path)
    )
    print(f"Loaded {filename}: {df.count():,} rows, {len(df.columns)} columns")
    return df

# Load all tables
df_basics    = load_tsv("title.basics.tsv.gz")
df_ratings   = load_tsv("title.ratings.tsv.gz")
df_crew      = load_tsv("title.crew.tsv.gz")
df_akas      = load_tsv("title.akas.tsv.gz")
df_principals= load_tsv("title.principals.tsv.gz")
df_episodes  = load_tsv("title.episode.tsv.gz")
df_names     = load_tsv("name.basics.tsv.gz")

Loaded title.basics.tsv.gz: 12,478,987 rows, 9 columns
Loaded title.ratings.tsv.gz: 1,668,663 rows, 3 columns
Loaded title.crew.tsv.gz: 12,478,987 rows, 3 columns
Loaded title.akas.tsv.gz: 57,013,962 rows, 8 columns
Loaded title.principals.tsv.gz: 99,275,168 rows, 6 columns
Loaded title.episode.tsv.gz: 9,637,883 rows, 4 columns
Loaded name.basics.tsv.gz: 15,297,466 rows, 6 columns


In [13]:
# ── Inspect schemas ──
print("=== title.basics ===")
df_basics.printSchema()
df_basics.show(3, truncate=False)

print("\n=== title.ratings ===")
df_ratings.printSchema()
df_ratings.show(3, truncate=False)

=== title.basics ===
root
 |-- tconst: string (nullable = true)
 |-- titleType: string (nullable = true)
 |-- primaryTitle: string (nullable = true)
 |-- originalTitle: string (nullable = true)
 |-- isAdult: string (nullable = true)
 |-- startYear: string (nullable = true)
 |-- endYear: string (nullable = true)
 |-- runtimeMinutes: string (nullable = true)
 |-- genres: string (nullable = true)

+---------+---------+----------------------+----------------------+-------+---------+-------+--------------+------------------------+
|tconst   |titleType|primaryTitle          |originalTitle         |isAdult|startYear|endYear|runtimeMinutes|genres                  |
+---------+---------+----------------------+----------------------+-------+---------+-------+--------------+------------------------+
|tt0000001|short    |Carmencita            |Carmencita            |0      |1894     |NULL   |1             |Documentary,Short       |
|tt0000002|short    |Le clown et ses chiens|Le clown et ses chiens

## 2. Data Cleaning & Filtering

In [14]:
# ── Step 2a: Filter by title type ──
VALID_TYPES = ["movie", "tvSeries", "tvMovie", "tvMiniSeries"]
df_basics_filtered = df_basics.filter(F.col("titleType").isin(VALID_TYPES))

print("Title type distribution (after filter):")
df_basics_filtered.groupBy("titleType").count().orderBy("count", ascending=False).show()

Title type distribution (after filter):
+------------+------+
|   titleType| count|
+------------+------+
|       movie|745163|
|    tvSeries|298867|
|     tvMovie|154790|
|tvMiniSeries| 69997|
+------------+------+



In [15]:
# ── Step 2b: Remove adult content ──
df_basics_filtered = df_basics_filtered.filter(F.col("isAdult") == "0")
print(f"After removing adult content: {df_basics_filtered.count():,} titles")

After removing adult content: 1,256,024 titles


In [16]:
# ── Step 2c: Cast numeric columns ──
df_basics_filtered = (
    df_basics_filtered
    .withColumn("startYear", F.col("startYear").cast(IntegerType()))
    .withColumn("endYear", F.col("endYear").cast(IntegerType()))
    .withColumn("runtimeMinutes", F.col("runtimeMinutes").cast(IntegerType()))
)

df_ratings = (
    df_ratings
    .withColumn("averageRating", F.col("averageRating").cast(FloatType()))
    .withColumn("numVotes", F.col("numVotes").cast(IntegerType()))
)

print("Numeric columns cast successfully.")

Numeric columns cast successfully.


In [17]:
# ── Step 2d: Drop rows with critical missing values ──
before = df_basics_filtered.count()
df_basics_clean = df_basics_filtered.dropna(subset=["startYear", "runtimeMinutes", "genres"])
after = df_basics_clean.count()
print(f"Dropped {before - after:,} rows with missing startYear/runtimeMinutes/genres")
print(f"Remaining: {after:,} titles")

Dropped 597,766 rows with missing startYear/runtimeMinutes/genres
Remaining: 658,258 titles


## 3. Join Tables

In [18]:
# ── Step 3a: Join basics with ratings ──
df_main = df_basics_clean.join(df_ratings, on="tconst", how="inner")
print(f"After joining with ratings: {df_main.count():,} titles")
df_main.show(3, truncate=False)

After joining with ratings: 415,324 titles
+---------+---------+-----------------------------+-----------------------------+-------+---------+-------+--------------+--------------------------+-------------+--------+
|tconst   |titleType|primaryTitle                 |originalTitle                |isAdult|startYear|endYear|runtimeMinutes|genres                    |averageRating|numVotes|
+---------+---------+-----------------------------+-----------------------------+-------+---------+-------+--------------+--------------------------+-------------+--------+
|tt0000009|movie    |Miss Jerry                   |Miss Jerry                   |0      |1894     |NULL   |45            |Romance                   |5.3          |240     |
|tt0000147|movie    |The Corbett-Fitzsimmons Fight|The Corbett-Fitzsimmons Fight|0      |1897     |NULL   |100           |Documentary,News,Sport    |5.3          |603     |
|tt0000574|movie    |The Story of the Kelly Gang  |The Story of the Kelly Gang  |0      |190

In [19]:
# ── Step 3b: Filter by numVotes (instructor feedback) ──
MIN_VOTES = 100
before = df_main.count()
df_main = df_main.filter(F.col("numVotes") >= MIN_VOTES)
after = df_main.count()
print(f"Filtered titles with numVotes < {MIN_VOTES}")
print(f"Removed: {before - after:,} titles")
print(f"Remaining: {after:,} titles with reliable ratings")

Filtered titles with numVotes < 100
Removed: 229,123 titles
Remaining: 186,201 titles with reliable ratings


In [20]:
# ── Step 3c: Join with crew (directors and writers) ──
df_main = df_main.join(df_crew, on="tconst", how="left")
print(f"After joining with crew: {df_main.count():,} titles")

After joining with crew: 186,201 titles


In [21]:
# ── Step 3d: Aggregate language info from title.akas ──
df_lang = (
    df_akas
    .filter(F.col("language").isNotNull())
    .groupBy("titleId", "language")
    .count()
    .withColumn(
        "rank",
        F.row_number().over(
            Window.partitionBy("titleId").orderBy(F.desc("count"))
        )
    )
    .filter(F.col("rank") == 1)
    .select(
        F.col("titleId").alias("tconst"),
        F.col("language").alias("primaryLanguage")
    )
)

df_region_count = (
    df_akas
    .filter(F.col("region").isNotNull())
    .groupBy("titleId")
    .agg(F.countDistinct("region").alias("numRegions"))
    .withColumnRenamed("titleId", "tconst")
)

df_main = df_main.join(df_lang, on="tconst", how="left")
df_main = df_main.join(df_region_count, on="tconst", how="left")
print(f"After joining with language/region info: {df_main.count():,} titles")

After joining with language/region info: 186,201 titles


In [22]:
# ── Step 3e: Aggregate principal cast/crew info ──
df_principals_agg = (
    df_principals
    .groupBy("tconst")
    .agg(
        F.count("nconst").alias("numPrincipals"),
        F.countDistinct("category").alias("numRoleTypes")
    )
)
df_main = df_main.join(df_principals_agg, on="tconst", how="left")
print(f"After joining with principals: {df_main.count():,} titles")

After joining with principals: 186,201 titles


## 4. Integrate User Ratings (Dataset.npy)

User ratings aggregation is done in **pandas** because PySpark's CSV reader
on Windows has issues with paths containing spaces. The aggregated result is
merged with the main dataset via pandas.

In [23]:
# ── Load and parse user ratings ──
print("Loading Dataset.npy (4.67M user ratings)...")
user_ratings_raw = np.load(os.path.join(DATA_DIR, "Dataset.npy"), allow_pickle=True)
print(f"Loaded {len(user_ratings_raw):,} user ratings")
print(f"Sample: {user_ratings_raw[:3]}")

parsed = []
for row in user_ratings_raw:
    parts = row.split(",")
    if len(parts) >= 3:
        parsed.append((parts[0].strip(), parts[1].strip(), int(parts[2].strip())))

pdf_user = pd.DataFrame(parsed, columns=["userId", "titleId", "userRating"])
print(f"\nParsed {len(pdf_user):,} ratings")
print(f"Unique users: {pdf_user['userId'].nunique():,}")
print(f"Unique titles: {pdf_user['titleId'].nunique():,}")
pdf_user.head()

Loading Dataset.npy (4.67M user ratings)...
Loaded 4,669,820 user ratings
Sample: ['ur4592644,tt0120884,10,16 January 2005'
 'ur3174947,tt0118688,3,16 January 2005'
 'ur3780035,tt0387887,8,16 January 2005']

Parsed 4,669,820 ratings
Unique users: 1,499,238
Unique titles: 351,109


,userId,titleId,userRating
0,ur4592644,tt0120884,10
1,ur3174947,tt0118688,3
2,ur3780035,tt0387887,8
3,ur4592628,tt0346491,1
4,ur3174947,tt0094721,8


In [24]:
# ── Aggregate user-level features per title (in pandas) ──
print("Aggregating user ratings per title...")
df_user_agg_pd = (
    pdf_user
    .groupby("titleId")
    .agg(
        userRatingCount=("userRating", "count"),
        userRatingMean=("userRating", "mean"),
        userRatingStd=("userRating", "std"),
        userRatingMin=("userRating", "min"),
        userRatingMax=("userRating", "max"),
    )
    .reset_index()
)

# Extreme rating count (ratings <= 2 or >= 9)
extreme_counts = (
    pdf_user
    .assign(is_extreme=lambda x: (x['userRating'] <= 2) | (x['userRating'] >= 9))
    .groupby("titleId")["is_extreme"]
    .sum()
    .reset_index()
    .rename(columns={"is_extreme": "extremeRatingCount"})
)

df_user_agg_pd = df_user_agg_pd.merge(extreme_counts, on="titleId")
df_user_agg_pd["extremeRatingRatio"] = df_user_agg_pd["extremeRatingCount"] / df_user_agg_pd["userRatingCount"]
df_user_agg_pd["userRatingRange"] = df_user_agg_pd["userRatingMax"] - df_user_agg_pd["userRatingMin"]
df_user_agg_pd.rename(columns={"titleId": "tconst"}, inplace=True)

print(f"Aggregated user features for {len(df_user_agg_pd):,} unique titles")
df_user_agg_pd.head()

Aggregating user ratings per title...
Aggregated user features for 351,109 unique titles


,tconst,userRatingCount,userRatingMean,userRatingStd,userRatingMin,userRatingMax,extremeRatingCount,extremeRatingRatio,userRatingRange
0,tt0000001,8,6.500000,2.828427,2,10,3,0.375000,8
1,tt0000003,13,7.076923,1.800997,3,10,2,0.153846,7
2,tt0000005,18,7.222222,2.669117,2,10,8,0.444444,8
3,tt0000007,3,6.000000,2.000000,4,8,0,0.000000,4
4,tt0000008,11,7.181818,2.960344,1,10,5,0.454545,9


In [25]:
# ── Merge user features with main dataset via pandas ──
print("Converting main Spark DataFrame to pandas for merge...")
pdf_main = df_main.toPandas()
print(f"Main dataset: {len(pdf_main):,} titles")

# Merge
pdf_main = pdf_main.merge(df_user_agg_pd, on="tconst", how="left")

# Fill nulls for titles without user ratings
user_cols = ["userRatingCount", "userRatingMean", "userRatingStd",
             "userRatingMin", "userRatingMax", "extremeRatingCount",
             "extremeRatingRatio", "userRatingRange"]
pdf_main[user_cols] = pdf_main[user_cols].fillna(0)

matched = (pdf_main['userRatingCount'] > 0).sum()
print(f"\nTitles WITH user ratings: {matched:,} / {len(pdf_main):,}")
print(f"Columns: {len(pdf_main.columns)}")
pdf_main[['tconst', 'primaryTitle', 'averageRating', 'userRatingCount', 'userRatingMean']].head(10)

Converting main Spark DataFrame to pandas for merge...
Main dataset: 186,201 titles

Titles WITH user ratings: 127,803 / 186,201
Columns: 25


,tconst,primaryTitle,averageRating,userRatingCount,userRatingMean
0,tt0003665,The Battle of the Sexes,6.3,2.0,8.000000
1,tt0003883,The Child of Paris,7.2,4.0,8.500000
2,tt0003973,A Florida Enchantment,5.8,6.0,5.333333
3,tt0004066,Sealed Orders,6.8,7.0,7.857143
4,tt0004134,Hypocrites,6.4,34.0,6.294118
5,tt0004635,The Squaw Man,5.6,8.0,5.000000
6,tt0004712,The Undesirable,6.2,0.0,0.000000
7,tt0004872,Alias Jimmy Valentine,6.5,2.0,6.500000
8,tt0004873,Alice in Wonderland,6.1,11.0,6.909091
9,tt0006864,Intolerance,7.6,93.0,8.118280


## 5. Summary & Save

In [26]:
# ── Summary statistics ──
print("=== Dataset Shape ===")
print(f"Rows: {len(pdf_main):,}")
print(f"Columns: {len(pdf_main.columns)}")

print("\n=== Numeric Summary ===")
print(pdf_main[['averageRating', 'numVotes', 'runtimeMinutes', 'startYear',
                'numRegions', 'numPrincipals', 'userRatingCount', 'userRatingMean']].describe())

print("\n=== Title Type Distribution ===")
print(pdf_main['titleType'].value_counts())

print("\n=== Language Distribution (Top 15) ===")
print(pdf_main['primaryLanguage'].value_counts().head(15))

=== Dataset Shape ===
Rows: 186,201
Columns: 25

=== Numeric Summary ===
       averageRating      numVotes  runtimeMinutes      startYear  \
count  186201.000000  1.862010e+05    1.862010e+05  186201.000000   
mean        6.087350  7.893623e+03    1.123020e+02    2000.783068   
std         1.298478  5.375405e+04    8.557559e+03      23.144424   
min         1.000000  1.000000e+02    1.000000e+00    1894.000000   
25%         5.300000  1.990000e+02    8.000000e+01    1990.000000   
50%         6.300000  4.670000e+02    9.200000e+01    2009.000000   
75%         7.000000  1.663000e+03    1.050000e+02    2018.000000   
max         9.900000  3.185031e+06    3.692080e+06    2026.000000   

          numRegions  numPrincipals  userRatingCount  userRatingMean  
count  186170.000000  186020.000000    186201.000000   186201.000000  
mean       13.125482      17.501387        21.632413        4.590534  
std        10.028091       4.659713       115.876196        3.476148  
min         1.000000 

In [27]:
# ── Save preprocessed data as CSV ──
output_path = os.path.join(OUTPUT_DIR, "parquet", "preprocessed")
os.makedirs(output_path, exist_ok=True)
save_path = os.path.join(output_path, "data.csv")
pdf_main.to_csv(save_path, index=False)

print(f"Preprocessed data saved to: {save_path}")
print(f"Total rows: {len(pdf_main):,}")
print(f"Total columns: {len(pdf_main.columns)}")


Preprocessed data saved to: outputs\parquet\preprocessed\data.csv
Total rows: 186,201
Total columns: 25


In [28]:
# ── Verify saved data ──
df_check = pd.read_csv(save_path)
print(f"Verification - loaded {len(df_check):,} rows")
print(f"User ratings check: max userRatingCount = {df_check['userRatingCount'].max()}")

Verification - loaded 186,201 rows
User ratings check: max userRatingCount = 10534.0


In [29]:
# Keep Spark session alive for next notebook (or stop here)
# spark.stop()
print("Preprocessing complete! Proceed to 02_feature_engineering.ipynb")

Preprocessing complete! Proceed to 02_feature_engineering.ipynb
